In [ ]:
gen_report = False
show_plots = False

In [2]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if "dir" in k.lower() or "split" in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace("experiments/", "").replace("/results", "")
        records.append(record)

df = pd.DataFrame(records)

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map(
    {True: "yes", False: "no"}
)
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map(
    {True: "yes", False: "no"}
)

df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
df.loc[df["agent_a_training_mode"] == "frozen", "learning_rate_phase2_a"] = "-"
df.loc[df["best_of_n"].isna(), "best_of_n"] = 1
df.loc[df["dataset_tt"].isna(), "dataset_tt"] = "one_shape"
df.loc[df["sampling_temperature_tt"].isna(), "sampling_temperature_tt"] = df["sampling_temperature"]
df.loc[df["gumbel"].isna(), "gumbel"] = False

df.loc[(df["num_iterations"] == 0) | (df["best_of_n"] == 1), "test_time_mode"] = "-"

df.rename(
    columns={"test_time_training_accuracy": "test_time_mutual_accuracy"}, inplace=True
)

df.rename(
    columns={"test_time_self_play_accuracy": "test_time_self_accuracy"}, inplace=True
)

df["test_time_mode"] = df["test_time_mode"].replace("adaptation", "sample_adaptation")
df["dataset"] = df["dataset"].replace("shape", "shape1")
df["dataset_tt"] = df["dataset_tt"].replace("two_shape", "shape2")
df["dataset_tt"] = df["dataset_tt"].replace(
    "shape_unique_double_attribute", "dual_attribute_shape"
)
# df["dataset_tt"] = df["dataset_tt"].replace("shape_unique_double_attribute", "single_attribute_shape")

df.loc[df["baseline"].isna(), "baseline"] = False

df.loc[
    df["baseline"]
    & (df["test_time_mode"] != "oracle_adaptation")
    & (df["test_time_mode"] != "oracle_full_adaptation"),
    "test_time_mode",
] = "-"

df["test_time_mutual_accuracy"] = df["test_time_mutual_accuracy"] * 100
df["test_time_self_accuracy"] = df["test_time_self_accuracy"] * 100
df["mutual_play_accuracy"] = df["mutual_play_accuracy"] * 100
df["self_play_accuracy_a"] = df["self_play_accuracy_a"] * 100

df = df.where(pd.notna(df), "None")
pd.options.display.float_format = "{:.10g}".format
df = df.map(
    lambda x: (
        f"{x:.0e}"
        if isinstance(x, (int, float))
        and x != 0
        and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01)
        else x
    )
)
print("Total reports loaded:", len(df))

pd.set_option("display.max_rows", None)

/tmp/ipykernel_1087749/99214765.py:43: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["num_iterations"] == 0, "learning_rate_tt"] = "-"
/tmp/ipykernel_1087749/99214765.py:44: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["agent_a_training_mode"] == "frozen", "learning_rate_phase2_a"] = "-"


Total reports loaded: 8649


In [ ]:
def filter_df(
    filters,
    df=df,
    sort_by=[
        "message_length",
        "message_length_tt",
        "learning_rate_tt",
        "num_iterations",
    ],
):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if (type(value) == str) and value.startswith(">"):
                mask |= df[key] > value.split(">")[1]
            if value == "!None":
                mask |= df[key] != "None"
            else:
                mask |= df[key] == value
        idx &= mask

    return df[idx].sort_values(by=sort_by).reset_index(drop=True)

In [ ]:
import os
import shutil


def clean_expr(res_df, check_before_delete=True):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        if check_before_delete:
            answer = (
                input(f"⚠️ Remove all files in '{path}' except report.json? (y/n): ")
                .strip()
                .lower()
            )
        else:
            answer = "y"

        if answer != "y":
            print(f"❌ Skipped: {path}")
            continue

        for item in os.listdir(full_path):
            print(item)
            if item == "results":
                continue

            item_path = os.path.join(full_path, item)

            try:
                if os.path.isdir(item_path):
                    shutil.rmtree(item_path)
                else:
                    os.remove(item_path)
            except Exception as e:
                print(f"[ERROR] {item_path}: {e}")

        print(f"✅ Cleaned: {path}")

In [ ]:
import base64


def to_html(df):
    styled = (
        df.style.hide(axis="index")
        .format(
            lambda x: (
                f"{int(x)}"  # 20.0 -> 20
                if isinstance(x, float) and x.is_integer()
                else (
                    f"{x:.0e}"  # small numbers -> scientific
                    if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01
                    else x
                )
            )
        )
        .set_table_styles(
            [
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {
                    "selector": "th",
                    "props": [
                        ("border-right", "1px solid black"),
                        ("color", "darkblue"),
                        ("font-weight", "bold"),
                        ("padding-left", "8px"),
                        ("padding-right", "8px"),
                    ],
                },
            ]
        )
        .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)


def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = [
        "#0b3d91",  # dark blue
        "#800000",  # maroon
        "#205522",  # dark green
        "#111011",  # indigo
        "#444444",
    ]  # dark gray
    color = colors[(num - 1) % len(colors)]  # cycle through colors

    if gen_report:
        with open("results.html", "a") as f:
            f.write(
                f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n'
            )


def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")


def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')


def write(text):
    html_text = text.replace("\n", "<br>\n")

    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,'
        + encoded
        + '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)

    if os.path.exists(path):
        os.remove(path)

In [ ]:
import os
import shutil


def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [ ]:
metrics = []

In [ ]:
import numpy as np
import pandas as pd
from itertools import product


def _find_elbow_idx(x, y):
    if len(x) <= 2:
        return np.argmax(y)

    x = np.asarray(x)
    y = np.asarray(y)

    x1, y1 = x[0], y[0]
    x2, y2 = x[-1], y[-1]

    numerator = np.abs((x2 - x1) * (y1 - y) - (y2 - y1) * (x1 - x))

    denominator = np.hypot(x2 - x1, y2 - y1)

    distances = numerator / denominator

    return np.argmax(distances)


def extract_maxes_elbow(
    df,
    cols=["message_length", "message_length_tt", "seed"],
    max_col="test_time_self_accuracy",
):
    values = []

    for col in cols:
        values.append(set(df[col]))

    combinations = [list(x) for x in product(*values)]

    maxes = []

    for comb in combinations:

        section = filter_df(
            {col: value for col, value in zip(cols, comb)},
            df,
        )

        if section.empty:
            continue

        # Split this section by learning rate
        elbow_rows = []

        for lr, lr_section in section.groupby("learning_rate_tt"):

            lr_section = lr_section.copy()

            lr_section["num_iterations"] = pd.to_numeric(lr_section["num_iterations"])

            lr_section[max_col] = pd.to_numeric(lr_section[max_col])

            lr_section = lr_section.sort_values("num_iterations")

            x = lr_section["num_iterations"].values
            y = lr_section[max_col].values

            elbow_idx = _find_elbow_idx(x, y)

            elbow_row = lr_section.iloc[elbow_idx]

            elbow_rows.append(elbow_row)

        if len(elbow_rows) == 0:
            continue

        elbow_df = pd.DataFrame(elbow_rows)

        # Among all learning rates, keep the elbow row with the
        # highest accuracy
        best_elbow_row = elbow_df.loc[elbow_df[max_col].idxmax()]

        maxes.append(best_elbow_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)

    return final_df.sort_values(by=cols).reset_index(drop=True)

In [ ]:
def extract_maxes(
    df,
    cols=["message_length", "message_length_tt", "seed"],
    max_col="test_time_mutual_accuracy",
):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[pd.to_numeric(section[max_col]).idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=cols).reset_index(drop=True)

In [ ]:
def find_best_length_elbow(df):
    x = df["message_length_tt"].to_numpy()

    y = (
        df["test_time_self_accuracy"]
        .str.extract(r"([\d.]+)")
        .astype(float)
        .squeeze()
        .to_numpy()
    )

    idx = _find_elbow_idx(x, y)
    return x[_find_elbow_idx(x, y)], df.loc[idx, "test_time_mutual_accuracy"]

In [ ]:
from itertools import product
import pandas as pd


def mean_and_std(
    df,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
    ],
    metrics=["test_time_self_accuracy", "test_time_mutual_accuracy"] + metrics,
    precision=1,
):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i : i + 3]

        rows.append(
            {col: chunk[col].iloc[0] for col in config_cols}
            |
            # {metric: f"{pd.to_numeric(chunk[metric]).mean() * 100:.1f} ± {pd.to_numeric(chunk[metric]).std() * 100:.1f}" for metric in metrics}
            {
                metric: (
                    None
                    if pd.to_numeric(chunk[metric], errors="coerce").dropna().empty
                    else f"{pd.to_numeric(chunk[metric], errors='coerce').mean():.{precision}f} ± "
                    f"{pd.to_numeric(chunk[metric], errors='coerce').std():.{precision}f}"
                )
                for metric in metrics
            }
        )

    out = pd.DataFrame(rows)
    return out.sort_values(by=config_cols)

In [ ]:
def smooth(values, factor=0.5):
    """Exponential moving average smoothing."""
    smoothed = []
    s = values[0]
    for v in values:
        s = factor * s + (1 - factor) * v
        smoothed.append(s)
    return smoothed

In [ ]:
import matplotlib.pyplot as plt

METHOD_COLORS = {
    "GS-ST": "#8c2d04",
    "REINFORCE": "#1f77b4",
    "VQEL": "#ff7f0e",
    "VQEL + TTA (Batch)": "#2ca02c",
    "VQEL + TTA (Dataset)": "#9467bd",
    "VQEL + TTA (Full)": "#7f7f7f",
    "VQEL + TTA (MG)": "#2ca02c",
    "VQEL + TTS": "#7f7f7f",
    "LR = 1e-4": "#8c2d04",
    "LR = 1e-5": "#7f7f7f",
    "LR = 1e-6": "#2ca02c",
    "Oracle": "#000000",
    "Oracle (MG)": "#000000",
    "Oracle (Full)": "#7f7f7f",
}


def plot(
    dfs,
    labels=(
        "GS-ST",
        "REINFORCE",
        "VQEL",
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle",
        "VQEL + TTS",
    ),
    name="name",
    xcol="message_length_tt",
    xlabel="Test-Time Message Length",
    marker="o",
    smooth_factor=0,
    xticks=None,
    logscale=False,
    base=10,
    grid=True,
):
    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)

    all_x = []

    for df, label in zip(dfs, labels):
        if label not in METHOD_COLORS:
            raise ValueError(f"No color defined for label: {label}")

        color = METHOD_COLORS[label]

        df = df.copy()

        # Split "mean ± std"
        df[["mean_acc", "std_acc"]] = (
            df["test_time_mutual_accuracy"].str.split("±", expand=True).astype(float)
        )

        df = df.sort_values(xcol)

        x = df[xcol]
        y = df["mean_acc"]
        err = df["std_acc"]

        all_x.extend(x.tolist())

        ax.plot(
            x,
            smooth(y, factor=smooth_factor),
            marker=marker if "Oracle" not in label else None,
            linestyle="-" if "Oracle" not in label else "--",
            linewidth=2,
            color=color,
            label=label,
        )

        ax.fill_between(
            x,
            smooth(y - err, factor=smooth_factor),
            smooth(y + err, factor=smooth_factor),
            color=color,
            alpha=0.2,
            linewidth=0,
        )

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel("OoD Accuracy (%)", fontsize=12)

    if xticks == None:
        xticks = sorted(set(all_x))
    ax.set_xticks(xticks)

    ax.tick_params(axis="both", labelsize=10)

    ax.legend(frameon=False)
    if grid:
        ax.grid(alpha=0.3)

    if logscale:
        ax.set_xscale("log", base=base)
        if base == 2:
            ax.set_xticklabels(xticks)
    fig.tight_layout()

    plt.savefig(
        f"assets/{name}.pdf",
        format="pdf",
        bbox_inches="tight",
    )

    if show_plots:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
clear()

---

In [ ]:
gumbel_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "tau_0",
    "mutual_play_accuracy",
    "sampling_temperature",
    "test_time_mode",
    "path",
]

backbone_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "test_time_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "path",
]

backbone_cols_report = [
    "seed",
    "message_length",
    "agent_a_training_mode",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

baseline_cols = (
    [
        "seed",
        "baseline",
        "dataset",
        "sim",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "test_time_self_accuracy",
        "test_time_mutual_accuracy",
    ]
    + metrics
    + [
        "path",
    ]
)

baseline_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

scaling_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

scaling_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "sampling_temperature_tt",
    "best_of_n",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

adapt_cols = [
    "seed",
    "baseline",
    "dataset",
    "sim",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

adapt_cols_report = [
    "seed",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
]

# Shape

## Gumbel - ID

In [ ]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "None",
        "baseline": True,
        "gumbel": True,
    },
    sort_by=["seed"],
)
# res[gumbel_cols]

In [ ]:
final = extract_maxes(res, max_col="mutual_play_accuracy")
# final[gumbel_cols]

In [ ]:
mean = mean_and_std(
    final, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
to_html(mean)
mean

## Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "!None",
        "baseline": True,
        "test_time_mode": "-",
        "gumbel": True,
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
gumbel = mean_and_std(
    res, metrics=["test_time_self_accuracy", "test_time_mutual_accuracy"]
)
gumbel

## REINFORCE - ID

In [ ]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "None",
        "baseline": True,
        "gumbel": False,
    },
    sort_by=["seed"],
)
# res[backbone_cols]

In [ ]:
mean = mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
to_html(mean)
mean

## REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dialogued_checkpoint": "!None",
        "baseline": True,
        "gumbel": False,
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
REINFORCE = mean_and_std(res)
REINFORCE

## VQEL - ID

In [ ]:
add_heading(2, "Shape1")
add_heading(3, "Base Model")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "baseline": False,
        "message_length": "[3, 4]",
        "test_time_mode": "-",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length"],
)
res = extract_maxes(res, max_col="mutual_play_accuracy")
res[backbone_cols]

In [ ]:
mean = mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)
to_html(mean)
mean

## VQEL - OOD

In [ ]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "dataset_tt": "shape2",
        "baseline": False,
        "message_length": "[3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)

to_html(res[baseline_cols_report])

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
VQ_NoTT

In [ ]:
find_best_length_elbow(VQ_NoTT)

## Scaling

In [ ]:
add_heading(3, "Scaling")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "VQEL": True,
        "message_length": "[3, 4]",
        "test_time_mode": "scaling",
    },
    sort_by=[
        "message_length",
        "message_length_tt",
        "seed",
        "sampling_temperature_tt",
        "best_of_n",
    ],
)

to_html(res[scaling_cols_report])

# res[scaling_cols]

In [ ]:
final = extract_maxes(res)
# final[scaling_cols]

In [ ]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [ ]:
add_heading(3, "Adaptation")

res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "VQEL": True,
        "message_length": "[3, 4]",
        "test_time_mode": ["batch_adaptation"],
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(final)
batch_adapt

In [ ]:
find_best_length_elbow(batch_adapt)

## Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "shape1",
        "sim": "cosine",
        "VQEL": True,
        "message_length": "[3, 4]",
        "test_time_mode": ["dataset_adaptation"],
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt"],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(final)
# dataset_adapt

In [ ]:
find_best_length_elbow(dataset_adapt)

## Oracle

In [ ]:
res = filter_df(
    {
        "dataset": "shape1",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "shape2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle = mean_and_std(final)
oracle

## Oracle Full

In [ ]:
res = filter_df(
    {
        "dataset": "shape1",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "shape2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle_full = mean_and_std(final)
oracle_full

## Plot

In [ ]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, dataset_adapt, oracle],
    name="main-results/shape",
)

In [ ]:
plot(
    [batch_adapt, dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/shape",
)

In [ ]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, scaling],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/shape",
)

# MNIST

## Gumbel - ID

In [ ]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "None",
        "gumbel": True,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["seed"],
)

# res[gumbel_cols]

In [ ]:
final = extract_maxes(res, max_col="mutual_play_accuracy")
final[gumbel_cols]

In [ ]:
mean = mean_and_std(
    final, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
to_html(mean)
mean

## Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "!None",
        "gumbel": True,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
mnist_gumbel = mean_and_std(res)
mnist_gumbel

## REINFORCE - ID

In [ ]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "None",
        "gumbel": False,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["seed"],
)
res[backbone_cols]

In [ ]:
mean_and_std(res)

## REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "dialogued_checkpoint": "!None",
        "gumbel": False,
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
mnist_REINFORCE = mean_and_std(res)
mnist_REINFORCE

## VQEL - ID

In [ ]:
add_heading(2, "MNIST")
add_heading(3, "Base Model")
res = filter_df(
    {
        "dataset": "mnist1",
        "sim": "cosine",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": "[3, 4]",
        "number_of_candidates": 100,
    },
    sort_by=["message_length", "message_length_tt"],
)
res[backbone_cols]

In [ ]:
mean = mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)
mean

## VQEL - OOD

In [ ]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "mnist1",
        "sim": "cosine",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "-",
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
final = extract_maxes(res)
# final[baseline_cols]

In [ ]:
mnist_VQ_NoTT = mean_and_std(final)
# mnist_VQ_NoTT

In [ ]:
find_best_length_elbow(mnist_VQ_NoTT)

## Scaling

In [ ]:
add_heading(3, "Scaling")
res = filter_df(
    {
        "dataset": "mnist1",
        "sim": "cosine",
        "baseline": False,
        "test_time_mode": "scaling",
        "sampling_temperature_tt": "1e-02",
        "number_of_candidates": 100,
    },
    sort_by=[
        "message_length",
        "message_length_tt",
        "sampling_temperature_tt",
        "best_of_n",
    ],
)
# res[scaling_cols]

In [ ]:
final = extract_maxes(res)
# final[scaling_cols]

In [ ]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [ ]:
add_heading(3, "Adaptation")

res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [ ]:
mnist_batch_adapt = mean_and_std(final)
mnist_batch_adapt

In [ ]:
find_best_length_elbow(mnist_batch_adapt)

## Dataset Adaptation

In [ ]:
add_heading(3, "Adaptation")

res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "number_of_candidates": 100,
        "message_length": "[3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)

res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res, max_col="test_time_self_accuracy")
# final[adapt_cols]

In [ ]:
mnist_dataset_adapt = mean_and_std(final)
mnist_dataset_adapt

In [ ]:
find_best_length_elbow(mnist_dataset_adapt)

## Oracle

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "mnist2",
        "number_of_candidates": 100,
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle = mean_and_std(final)
oracle

## Oracle Full

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "mnist2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle_full = mean_and_std(final)
oracle_full

## Plot

In [ ]:
plot(
    [
        mnist_gumbel,
        mnist_REINFORCE,
        mnist_VQ_NoTT,
        mnist_batch_adapt,
        mnist_dataset_adapt,
        oracle,
    ],
    name="main-results/mnist",
)

In [ ]:
plot(
    [mnist_batch_adapt, mnist_dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/mnist",
)

In [ ]:
plot(
    [mnist_gumbel, mnist_REINFORCE, mnist_VQ_NoTT, mnist_batch_adapt, scaling],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/mnist",
)

# ImageNet

## Gumbel - ID

In [ ]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
        "gumbel": True,
    },
    sort_by=["message_length", "seed"],
)
# res[gumbel_cols]

In [ ]:
final = extract_maxes(res, max_col="mutual_play_accuracy")
final[gumbel_cols]

In [ ]:
mean = mean_and_std(
    final, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)
mean

## Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "dialogued_checkpoint": "!None",
        "gumbel": True,
        "dataset_tt": "imagenet_same_class",
        "message_length": "[2, 3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
imagenet_gumbel = mean_and_std(
    res, metrics=["test_time_self_accuracy", "test_time_mutual_accuracy"] + metrics
)
imagenet_gumbel

## REINFORCE - ID

In [ ]:
add_heading(3, "Baseline")

res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
        "gumbel": False,
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

In [ ]:
mean_and_std(res)

## REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "!None",
        "gumbel": False,
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
final = extract_maxes(res)
# final[baseline_cols]

In [ ]:
imagenet_REINFORCE = mean_and_std(final)
imagenet_REINFORCE

## VQEL - ID

In [ ]:
add_heading(2, "ImageNet")
add_heading(3, "Base Model")

res = filter_df(
    {
        "dataset": "imagenet",
        "sim": "cosine",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": "[2, 3, 4]",
    },
    sort_by=["vocab_size", "message_length", "seed", "message_length_tt"],
)
res[backbone_cols]

In [ ]:
mean = mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)
# mean

## VQEL - OOD

In [ ]:
add_heading(3, "Baseline")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": "-",
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
imagenet_VQ_NoTT = mean_and_std(res)
# imagenet_VQ_NoTT

In [ ]:
find_best_length_elbow(imagenet_VQ_NoTT)

## Scaling

In [ ]:
add_heading(3, "Scaling")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": "scaling",
    },
    sort_by=["message_length_tt", "sampling_temperature_tt", "best_of_n"],
)
# res[scaling_cols]

In [ ]:
final = extract_maxes(res)
# final[scaling_cols]

In [ ]:
scaling = mean_and_std(final)
scaling

## Batch Adaptation

In [ ]:
add_heading(3, "Adaptation")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": ["batch_adaptation"],
        "message_length": "[2, 3, 4]",
        "num_iterations": [50, 100, 200],
    },
    sort_by=[
        "test_time_mode",
        "message_length_tt",
        "seed",
        "learning_rate_tt",
        "num_iterations",
    ],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res, max_col="test_time_mutual_accuracy")
# final[adapt_cols]

In [ ]:
imagenet_batch_adapt = mean_and_std(final)
# imagenet_batch_adapt

In [ ]:
find_best_length_elbow(imagenet_batch_adapt)

## Dataset Adaptation

In [ ]:
add_heading(3, "Adaptation")
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": ["dataset_adaptation"],
        "message_length": "[2, 3, 4]",
        "num_iterations": [5, 10, 15, 20],
    },
    sort_by=[
        "test_time_mode",
        "message_length_tt",
        "seed",
        "learning_rate_tt",
        "num_iterations",
    ],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res)
final[adapt_cols]

In [ ]:
imagenet_dataset_adapt = mean_and_std(final)
imagenet_dataset_adapt

In [ ]:
find_best_length_elbow(imagenet_dataset_adapt)

## Oracle

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle = mean_and_std(final)
oracle

## Oracle Full

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle_full = mean_and_std(final)
oracle_full

## Plot

In [ ]:
plot(
    [
        imagenet_gumbel,
        imagenet_REINFORCE,
        imagenet_VQ_NoTT,
        imagenet_batch_adapt,
        imagenet_dataset_adapt,
        oracle,
    ],
    name="main-results/imagenet",
)

In [ ]:
plot(
    [imagenet_batch_adapt, imagenet_dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/imagenet",
)

In [ ]:
plot(
    [
        imagenet_gumbel,
        imagenet_REINFORCE,
        imagenet_VQ_NoTT,
        imagenet_batch_adapt,
        scaling,
    ],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/imagenet",
)

# COCO

## Gumbel - ID

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": True,
        "gumbel": True,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length", "seed"],
)
# res[gumbel_cols]

In [ ]:
final = extract_maxes(res)
final[gumbel_cols]

In [ ]:
mean_and_std(final)

## Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": True,
        "gumbel": True,
        "dialogued_checkpoint": "!None",
        "dataset_tt": "coco_complex",
        "message_length": "[2, 3, 4]",
        "test_time_mode": "-",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
gumbel = mean_and_std(res)
gumbel

## REINFORCE - ID

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": True,
        "gumbel": False,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "None",
    },
    sort_by=["message_length", "seed"],
)
res[backbone_cols]

In [ ]:
final = extract_maxes(res)
final[backbone_cols]

In [ ]:
mean_and_std(final)

## REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": True,
        "gumbel": False,
        "message_length": "[2, 3, 4]",
        "dialogued_checkpoint": "!None",
    },
    sort_by=["message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
REINFORCE = mean_and_std(res)
REINFORCE

## VQEL - ID

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco",
        "baseline": False,
        "message_length": "[2, 3, 4]",
    },
    sort_by=["seed"],
)
res[backbone_cols]

In [ ]:
final = extract_maxes(res)
final[backbone_cols]

In [ ]:
mean_and_std(final)

## VQEL - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": False,
        "test_time_mode": "-",
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)
# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
VQ_NoTT

In [ ]:
find_best_length_elbow(VQ_NoTT)

## Scaling

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": False,
        "message_length": "[2, 3, 4]",
        "test_time_mode": "scaling",
    },
    sort_by=["message_length_tt", "seed", "sampling_temperature_tt"],
)
# res[scaling_cols]

In [ ]:
final = extract_maxes(res)
# final[scaling_cols]

In [ ]:
scaling = mean_and_std(final)
# scaling

## Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": False,
        "test_time_mode": ["batch_adaptation"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(final)
batch_adapt

In [ ]:
find_best_length_elbow(batch_adapt)

## Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "dataset_tt": "coco_complex",
        "baseline": False,
        "test_time_mode": ["dataset_adaptation"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "learning_rate_tt", "num_iterations"],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res)
# final[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(final)
dataset_adapt

In [ ]:
find_best_length_elbow(dataset_adapt)

## Oracle

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "test_time_mode": "oracle_adaptation",
        "dataset_tt": "coco_complex",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle = mean_and_std(final)
oracle

## Oracle Full

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "test_time_mode": "oracle_full_adaptation",
        "dataset_tt": "coco_complex",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)

In [ ]:
oracle_full = mean_and_std(final)
oracle_full

## Plot

In [ ]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, dataset_adapt, oracle],
    name="main-results/coco",
)

In [ ]:
plot(
    [batch_adapt, dataset_adapt, oracle, oracle_full],
    labels=(
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle (MG)",
        "Oracle (Full)",
    ),
    name="oracle/coco",
)

In [ ]:
plot(
    [gumbel, REINFORCE, VQ_NoTT, batch_adapt, scaling],
    labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Batch)", "VQEL + TTS"),
    name="scaling/coco",
)

# Batch Size

In [ ]:
adapt_cols = [
    "seed",
    "baseline",
    "dataset",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "number_of_candidates",
    "learning_rate_tt",
    "num_iterations",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

baseline_cols = [
    "seed",
    "baseline",
    "dataset",
    "message_length",
    "message_length_tt",
    "dataset_tt",
    "test_time_mode",
    "number_of_candidates",
    "test_time_self_accuracy",
    "test_time_mutual_accuracy",
    "path",
]

## ImageNet

In [ ]:
number_of_candidates = [2, 4, 8, 16, 32, 64, 118]

### Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "message_length_tt": 6,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=["message_length_tt", "number_of_candidates", "seed"],
)

# res[baseline_cols]

In [ ]:
gumbel = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "message_length_tt": 7,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=["message_length_tt", "number_of_candidates", "seed"],
)

# res[baseline_cols]

In [ ]:
REINFORCE = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### VQEL - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": False,
        "test_time_mode": "-",
        "message_length_tt": 6,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)

# res[adapt_cols]

In [ ]:
vqel = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)
# vqel

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "number_of_candidates": number_of_candidates,
        "message_length_tt": 5,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "number_of_candidates": number_of_candidates,
        "message_length_tt": 5,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Oracle

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_dog_breed",
        "test_time_mode": "oracle_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
res[adapt_cols]

In [ ]:
final = extract_maxes(res, cols=["number_of_candidates", "seed"])

In [ ]:
oracle = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)
oracle

### Plot

In [ ]:
plot(
    [gumbel, REINFORCE, vqel, batch_adapt, dataset_adapt, oracle],
    labels=[
        "GS-ST",
        "REINFORCE",
        "VQEL",
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle",
    ],
    xlabel="Number of Test-Time Distractors",
    xcol="number_of_candidates",
    logscale=True,
    base=2,
    name="batch/imagenet",
    xticks=[0, 0, 1, 3, 7, 15, 31, 63, 127],
)

## MNIST

In [ ]:
number_of_candidates = [2, 4, 8, 16, 32, 64, 128]

### Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "message_length_tt": 4,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [ ]:
gumbel = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "message_length_tt": 5,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [ ]:
REINFORCE = mean_and_std(
    res,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### VQEL - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "-",
        "message_length_tt": 7,
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [ ]:
vqel = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes_elbow(res, cols=["number_of_candidates", "seed"])
final[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)

### Oracle

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "test_time_mode": "oracle_adaptation",
        "number_of_candidates": number_of_candidates,
    },
    sort_by=[
        "message_length_tt",
        "number_of_candidates",
        "learning_rate_tt",
        "num_iterations",
        "seed",
    ],
)
# res[adapt_cols]

In [ ]:
final = extract_maxes(res, cols=["number_of_candidates", "seed"])
final[adapt_cols]

In [ ]:
oracle = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "number_of_candidates",
    ],
)
oracle

### Plot

In [ ]:
plot(
    [gumbel, REINFORCE, vqel, batch_adapt, dataset_adapt, oracle],
    labels=[
        "REINFORCE",
        "GS-ST",
        "VQEL",
        "VQEL + TTA (Batch)",
        "VQEL + TTA (Dataset)",
        "Oracle",
    ],
    xlabel="Number of Test-Time Distractors",
    xcol="number_of_candidates",
    logscale=True,
    base=2,
    name="batch/mnist",
    xticks=[0, 0, 1, 3, 7, 15, 31, 63, 127],
)

# Learning Rate & Steps

In [ ]:
num_iterations = (
    list(range(1, 10)) + list(range(10, 100, 10)) + list(range(100, 1000, 100)) + [1000]
)

## MNIST

In [ ]:
no_adapt = filter_df(
    {
        "dataset": "mnist1",
        "dataset_tt": "mnist2",
        "baseline": False,
        "test_time_mode": ["-"],
        "message_length_tt": 10,
        "learning_rate_tt": "-",
        "number_of_candidates": 100,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
# no_adapt[adapt_cols]

### Batch Adaptation

#### LR = 1e-4

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": ["1e-04"],
        "number_of_candidates": 100,
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### LR = 1e-5

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": ["1e-05"],
        "number_of_candidates": 100,
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr5

#### LR = 1e-6

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 10,
        "number_of_candidates": 100,
        "learning_rate_tt": ["1e-06"],
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [ ]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### Plot

In [ ]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=True, xticks=range(0, 1010, 100), smooth_factor=0.5, name="steps/mnist_steps_batch")

### Dataset Adaptation

#### LR = 1e-4

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": "1e-04",
        "number_of_candidates": 100,
        "path": ">20260300",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr4

#### LR = 1e-5

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": "1e-05",
        "number_of_candidates": 100,
        "path": ">20260300",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### LR = 1e-6

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 10,
        "learning_rate_tt": "1e-06",
        "number_of_candidates": 100,
        "path": ">20260300",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### Plot

In [ ]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=False, smooth_factor=0.5, xticks=range(0, 201, 25), name="steps/mnist_steps_dataset")

## ImageNet

In [ ]:
no_adapt = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "baseline": False,
        "test_time_mode": ["-"],
        "message_length_tt": 5,
        "learning_rate_tt": ["-"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
# no_adapt[adapt_cols]

### Batch Adaptation

#### LR = 1e-4

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": ["1e-04"],
        "message_length": "[2, 3, 4]",
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)

#### LR = 1e-5

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "dataset_tt": "imagenet_same_class",
        "VQEL": True,
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": "1e-05",
        "message_length": "[2, 3, 4]",
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [ ]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr5

#### LR = 1e-6

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "dataset_tt": "imagenet_same_class",
        "test_time_mode": ["batch_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": "1e-06",
        "message_length": "[2, 3, 4]",
        "path": ">20260300",
        "num_iterations": num_iterations,
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [ ]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr6

#### Plot

In [ ]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=True, xticks=range(0, 1010, 100), smooth_factor=0.5, name="steps/imagenet_steps_batch")

### Dataset Adaptation

#### LR = 1e-4

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "dataset_tt": "imagenet_same_class",
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": ["1e-04"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr4 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr4

#### LR = 1e-5

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "dataset_tt": "imagenet_same_class",
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 5,
        "learning_rate_tt": ["1e-05"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])
# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)
# final[adapt_cols]

In [ ]:
lr5 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr5

#### LR = 1e-6

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "VQEL": True,
        "test_time_mode": ["dataset_adaptation"],
        "message_length_tt": 5,
        "dataset_tt": "imagenet_same_class",
        "learning_rate_tt": ["1e-06"],
        "message_length": "[2, 3, 4]",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)
res = pd.concat([no_adapt, res])

# res[adapt_cols]

In [ ]:
final = extract_maxes(
    res, cols=["num_iterations", "seed"], max_col="test_time_mutual_accuracy"
)

In [ ]:
lr6 = mean_and_std(
    final,
    config_cols=[
        "dataset",
        "message_length",
        "message_length_tt",
        "dataset_tt",
        "test_time_mode",
        "num_iterations",
    ],
)
# lr6

#### Plot

In [ ]:
# plot([lr4, lr5, lr6], labels=("LR = 1e-4", "LR = 1e-5", "LR = 1e-6"), xcol="num_iterations", xlabel="Steps", marker=None, logscale=False, smooth_factor=0.6, xticks=range(0, 201, 25), name="steps/imagenet_steps_dataset")

# Computation Costs

In [ ]:
mask = df["inference_time"] != "None"
df.loc[mask, "inference_time"] = df.loc[mask, "inference_time"] / (1000)  # S

mask = df["peak_memory"] != "None"
df.loc[mask, "peak_memory"] = df.loc[mask, "peak_memory"] / (1024**3)  # GB

mask = df["flops"] != "None"
df.loc[mask, "flops"] = df.loc[mask, "flops"] / (10**9)  # GFLOPS

In [ ]:
computation_metrics = [
    "inference_time",
    "peak_memory",
    "flops",
]

## MNIST

### Gumbel

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": True,
        "message_length_tt": 10,
        "flops": "!None",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)

res[baseline_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### REINFORCE

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": False,
        "message_length_tt": 10,
        "flops": "!None",
    }
)

res[baseline_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### VQEL

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 10,
        "test_time_mode": "-",
        "flops": "!None",
    }
)

res[baseline_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 10,
        "test_time_mode": "batch_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 10,
        "test_time_mode": "dataset_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

## ImageNet

### Gumbel

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": True,
        "message_length_tt": 5,
        "flops": "!None",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)

res[baseline_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### REINFORCE

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": False,
        "message_length_tt": 5,
        "flops": "!None",
    },
    sort_by=["message_length_tt", "seed", "num_iterations"],
)

res[baseline_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### VQEL

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "test_time_mode": "-",
        "message_length_tt": 5,
        "flops": "!None",
    }
)

res[baseline_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 5,
        "test_time_mode": "batch_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "message_length_tt": 5,
        "test_time_mode": "dataset_adaptation",
        "flops": "!None",
    }
)

res[adapt_cols + computation_metrics]

In [ ]:
mean_and_std(res, metrics=["test_time_mutual_accuracy"] + computation_metrics)

# MPs Alignment

In [ ]:
alignment_metrics = [
    "mp_similarity",
    "mp_similarity_baseline_mean",
    "mp_similarity_baseline_std",
    "mp_similarity_p_value",
]

## Shape

In [ ]:
res = filter_df(
    {
        "dataset": "shape1",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

In [ ]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

## MNIST

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

In [ ]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

## ImageNet

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

In [ ]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

## COCO

In [ ]:
res = filter_df(
    {
        "dataset": "coco",
        "baseline": False,
        "gumbel": False,
        "mp_similarity": "!None",
    }
)

res[backbone_cols + alignment_metrics]

In [ ]:
mean_and_std(
    res, metrics=["test_time_mutual_accuracy"] + alignment_metrics, precision=3
)

# Random Length Trick

## MNIST

### Gumbel

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

# res[gumbel_cols]

In [ ]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

### Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [ ]:
gumbel = mean_and_std(res)
# gumbel

In [ ]:
gumbel3 = gumbel.iloc[:13].reset_index(drop=True)
gumbel4 = gumbel.iloc[13:].reset_index(drop=True)

### REINFORCE - ID

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

# res[backbone_cols]

In [ ]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

### REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [ ]:
REINFORCE = mean_and_std(res)
# REINFORCE

In [ ]:
REINFORCE3 = REINFORCE.iloc[:13].reset_index(drop=True)
REINFORCE4 = REINFORCE.iloc[13:].reset_index(drop=True)

### VQEL - ID

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist1",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

In [ ]:
mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)

### VQEL - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
# VQ_NoTT

In [ ]:
VQ_NoTT3 = VQ_NoTT.iloc[:13].reset_index(drop=True)
VQ_NoTT4 = VQ_NoTT.iloc[13:].reset_index(drop=True)

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(res)
# batch_adapt

In [ ]:
batch_adapt3 = batch_adapt.iloc[:13].reset_index(drop=True)
batch_adapt4 = batch_adapt.iloc[13:].reset_index(drop=True)

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "dataset_tt": "mnist2",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(res)
# dataset_adapt

In [ ]:
dataset_adapt3 = dataset_adapt.iloc[:13].reset_index(drop=True)
dataset_adapt4 = dataset_adapt.iloc[13:].reset_index(drop=True)

### Plot

In [ ]:
plot(
    [gumbel3, REINFORCE3, VQ_NoTT3, batch_adapt3, dataset_adapt3],
    name="rlt/mnist_without_rlt_l3",
)

In [ ]:
plot(
    [gumbel4, REINFORCE4, VQ_NoTT4, batch_adapt4, dataset_adapt4],
    name="rlt/mnist_without_rlt_l4",
)

## ImageNet

### Gumbel - ID

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[gumbel_cols]

In [ ]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

### Gumbel - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": True,
        "test_time_mode": "-",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [ ]:
gumbel = mean_and_std(res)
# gumbel

In [ ]:
gumbel3 = gumbel.iloc[:13].reset_index(drop=True)
gumbel4 = gumbel.iloc[13:].reset_index(drop=True)

### REINFORCE - ID

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

In [ ]:
mean_and_std(
    res, config_cols=["dataset", "message_length"], metrics=["mutual_play_accuracy"]
)

### REINFORCE - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": True,
        "gumbel": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [ ]:
REINFORCE = mean_and_std(res)
# REINFORCE

In [ ]:
REINFORCE3 = REINFORCE.iloc[:13].reset_index(drop=True)
REINFORCE4 = REINFORCE.iloc[13:].reset_index(drop=True)

### VQEL - ID

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "seed"],
)

res[backbone_cols]

In [ ]:
mean_and_std(
    res,
    config_cols=["dataset", "message_length"],
    metrics=["self_play_accuracy_a", "mutual_play_accuracy"],
)

### VQEL - OOD

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "-",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[baseline_cols]

In [ ]:
VQ_NoTT = mean_and_std(res)
VQ_NoTT

In [ ]:
VQ_NoTT3 = VQ_NoTT.iloc[:13].reset_index(drop=True)
VQ_NoTT4 = VQ_NoTT.iloc[13:].reset_index(drop=True)

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "batch_adaptation",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

res[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(res)
# batch_adapt

In [ ]:
batch_adapt3 = batch_adapt.iloc[:13].reset_index(drop=True)
batch_adapt4 = batch_adapt.iloc[13:].reset_index(drop=True)

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "dataset_adaptation",
        "dataset_tt": "imagenet_same_class",
        "message_length": ["[3]", "[4]"],
    },
    sort_by=["message_length", "message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(res)
# dataset_adapt

In [ ]:
dataset_adapt3 = dataset_adapt.iloc[:13].reset_index(drop=True)
dataset_adapt4 = dataset_adapt.iloc[13:].reset_index(drop=True)

### Plot

In [ ]:
plot(
    [gumbel3, REINFORCE3, VQ_NoTT3, batch_adapt3, dataset_adapt3],
    name="rlt/imagenet_without_rlt_l3",
)

In [ ]:
plot(
    [gumbel4, REINFORCE4, VQ_NoTT4, batch_adapt4, dataset_adapt4],
    name="rlt/imagenet_without_rlt_l4",
)

# Full sender Adaptation

## MNIST

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "full_sender_batch_adaptation",
        "dataset_tt": "mnist2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(final)
# batch_adapt

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "mnist1",
        "baseline": False,
        "test_time_mode": "full_sender_dataset_adaptation",
        "dataset_tt": "mnist2",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(res)
# dataset_adapt

### Plot

In [ ]:
# plot([mnist_gumbel, mnist_REINFORCE, mnist_VQ_NoTT, batch_adapt, mnist_batch_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/mnist_batch")

In [ ]:
# plot([mnist_gumbel, mnist_REINFORCE, mnist_VQ_NoTT, dataset_adapt, mnist_dataset_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/mnist_dataset")

## ImageNet

### Batch Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "full_sender_batch_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
final = extract_maxes(res)
# final[adapt_cols]

In [ ]:
batch_adapt = mean_and_std(final)
# batch_adapt

### Dataset Adaptation

In [ ]:
res = filter_df(
    {
        "dataset": "imagenet",
        "baseline": False,
        "test_time_mode": "full_sender_dataset_adaptation",
        "dataset_tt": "imagenet_same_class",
    },
    sort_by=["message_length_tt", "seed"],
)

# res[adapt_cols]

In [ ]:
dataset_adapt = mean_and_std(res)
dataset_adapt

### Plot

In [ ]:
# plot([imagenet_gumbel, imagenet_REINFORCE, imagenet_VQ_NoTT, batch_adapt, imagenet_batch_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/imagenet_batch")

In [ ]:
# plot([imagenet_gumbel, imagenet_REINFORCE, imagenet_VQ_NoTT, dataset_adapt, imagenet_dataset_adapt],
#       labels=("GS-ST", "REINFORCE", "VQEL", "VQEL + TTA (Full)", "VQEL + TTA (MG)"),
#       name="full_adaptation/imagenet_dataset")